# Compare MuscleMap WB + MedSAM (bbox-prompted) to Ground Truth

Evaluates MuscleMap WB segmentations refined by MedSAM (using MuscleMap's own
boxes as prompts) against myosegmenTUM ground truth. Water modality only —
mirrors `dafne_and_medsam/codes/column_compare_dafne_medsam.ipynb`.

In [1]:
import glob
import os
import re
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [2]:
BOUNDARY_DISTANCE = 1

EVAL_DIR = r'C:\Projects\dissector\eval_notebooks'
BASE_DIR = os.path.join(EVAL_DIR, 'muscle_map_wb_boxes_medsam')
GT_BASE  = os.path.join(EVAL_DIR, 'myosegmenTUM')

# (muscle_name, gt_label_int, npz_key) — same combined_gt convention used everywhere else
MUSCLES = [
    ('R_gracilis',  5, 'Gracilis_R'),
    ('L_gracilis',  1, 'Gracilis_L'),
    ('R_sartorius', 8, 'Sartorius_R'),
    ('L_sartorius', 4, 'Sartorius_L'),
]

print('GT_BASE:', os.path.abspath(GT_BASE))

GT_BASE: C:\Projects\dissector\eval_notebooks\myosegmenTUM


In [3]:
def parse_subject_stack(filename, modality_tag, npz_suffix):
    pattern = rf'^(.+)_{modality_tag}_stack(\d+)_{npz_suffix}\.npz$'
    m = re.match(pattern, os.path.basename(filename))
    if not m:
        return None, None
    return m.group(1), m.group(2)


def evaluate_muscle(muscle_name, gt_label_idx, npz_key,
                    seg_files, modality_tag, npz_suffix, result_dir, csv_suffix):
    results = []
    for seg_file in seg_files:
        subject, stack_num = parse_subject_stack(seg_file, modality_tag, npz_suffix)
        if subject is None:
            print(f'  could not parse: {os.path.basename(seg_file)}, skipping')
            continue

        gt_path = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                               f'combined_gt_stack{stack_num}.mha')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label_idx, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        data = np.load(seg_file)
        if npz_key not in data.files:
            print(f'  key "{npz_key}" not in {os.path.basename(seg_file)}, skipping')
            print(f'  available keys: {data.files}')
            continue
        pred_arr = data[npz_key].astype(np.uint8)

        pred_sitk = sitk.GetImageFromArray(pred_arr)
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            print(f'  {os.path.basename(seg_file)}: empty mask '
                  f'(gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
            hd = np.nan

        results.append({
            'subject':                              subject,
            'stack':                                stack_num,
            'pred_file':                            os.path.basename(seg_file),
            'gt_path':                              gt_path,
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df = pd.DataFrame(results)
    os.makedirs(result_dir, exist_ok=True)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_{csv_suffix}.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows -> {csv_path}')
    return df


print('Functions ready.')

Functions ready.


## Water

In [4]:
W_SEG_DIR    = os.path.join(BASE_DIR, 'segs_water')
W_RESULT_DIR = os.path.join(BASE_DIR, 'results_water')
NPZ_SUFFIX   = 'mm_medsam'

seg_files_w = sorted(glob.glob(os.path.join(W_SEG_DIR, f'*_{NPZ_SUFFIX}.npz')))
print(f'Seg dir   : {os.path.abspath(W_SEG_DIR)}')
print(f'Result dir: {os.path.abspath(W_RESULT_DIR)}')
print(f'Found     : {len(seg_files_w)} npz files')
for f in seg_files_w[:3]:
    print(' ', os.path.basename(f))

Seg dir   : C:\Projects\dissector\eval_notebooks\muscle_map_wb_boxes_medsam\segs_water
Result dir: C:\Projects\dissector\eval_notebooks\muscle_map_wb_boxes_medsam\results_water
Found     : 46 npz files
  HV001_1_WATER_stack1_mm_medsam.npz
  HV001_1_WATER_stack2_mm_medsam.npz
  HV001_2_WATER_stack1_mm_medsam.npz


In [5]:
dfs_w = {}
for muscle_name, gt_idx, npz_key in MUSCLES:
    print(f'\n── {muscle_name} (gt={gt_idx}, key="{npz_key}") ──')
    dfs_w[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, npz_key,
        seg_files_w, 'WATER', NPZ_SUFFIX, W_RESULT_DIR,
        csv_suffix='mm_medsam_bbox_water',
    )
print('\nDone.')


── R_gracilis (gt=5, key="Gracilis_R") ──


  Saved 46 rows -> C:\Projects\dissector\eval_notebooks\muscle_map_wb_boxes_medsam\results_water\df_R_gracilis_mm_medsam_bbox_water.csv

── L_gracilis (gt=1, key="Gracilis_L") ──


  Saved 46 rows -> C:\Projects\dissector\eval_notebooks\muscle_map_wb_boxes_medsam\results_water\df_L_gracilis_mm_medsam_bbox_water.csv

── R_sartorius (gt=8, key="Sartorius_R") ──


  Saved 46 rows -> C:\Projects\dissector\eval_notebooks\muscle_map_wb_boxes_medsam\results_water\df_R_sartorius_mm_medsam_bbox_water.csv

── L_sartorius (gt=4, key="Sartorius_L") ──


  Saved 46 rows -> C:\Projects\dissector\eval_notebooks\muscle_map_wb_boxes_medsam\results_water\df_L_sartorius_mm_medsam_bbox_water.csv

Done.


In [6]:
for name, df in dfs_w.items():
    print(f'\n── {name} ──')
    print(f'  n={len(df)}  mean_dice={df[f"{name}_dice"].mean():.4f}')


── R_gracilis ──
  n=46  mean_dice=0.7316

── L_gracilis ──
  n=46  mean_dice=0.7357

── R_sartorius ──
  n=46  mean_dice=0.7142

── L_sartorius ──
  n=46  mean_dice=0.7650
